In [1]:
#essential modules 
import os        # File and directory operations
import re        # Regular expressions for text cleanup
import time      # Sleep / wait utilities
import sqlite3   # Built-in SQLite database interface

# Modules for HTTP and HTML parsing
import requests
from bs4 import BeautifulSoup

# Selenium modules for real browser automation- for JavaScript
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

#webdriver-manager: auto-downloads matching ChromeDriver
from webdriver_manager.chrome import ChromeDriverManager

print("All libraries imported successfully.")
print(os.path.exists("StocksDatabase.db"))

# Output file names 
TEXT_FILE = "stocks.txt"         # Comma-separated output file
DB_FILE   = "StocksDatabase.db"  # SQLite database file
DB_TABLE  = "StocksTable"        # Table name inside the database

# Source URLs
CNN_MARKETS_URL  = "https://www.cnn.com/markets"       # Most Actives source
YAHOO_QUOTE_BASE = "https://finance.yahoo.com/quote/"  # Yahoo Finance base URL

# HTTP headers for Yahoo Finance
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# Timing constants
SELENIUM_WAIT = 15   # Max seconds to wait for JS elements to appear
CRAWL_DELAY   = 1.5  # Seconds between Yahoo Finance requests

print(f"Text file will be saved as    : {TEXT_FILE}")
print(f"Database file will be saved as: {DB_FILE}")


def build_chrome_driver():
    """
    Create and return a headless Chrome WebDriver.
    webdriver-manager downloads the correct ChromeDriver binary automatically.
    Opens CNN like a real browser so JavaScript loads the stock data.
    """
    chrome_options = Options()
    chrome_options.add_argument("--headless")              # No visible browser window
    chrome_options.add_argument("--no-sandbox")            # Required in some Linux/CI environments
    chrome_options.add_argument("--disable-dev-shm-usage") # Prevents shared-memory crashes
    chrome_options.add_argument("--disable-gpu")           # GPU not needed in headless mode
    chrome_options.add_argument("--window-size=1920,1080") # Full viewport so all elements render
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ) 

    # ChromeDriverManager().install() returns the path to the matching chromedriver binary
    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=chrome_options)
    return driver


def get_most_active_tickers():
    """
    Use Selenium to scrape 'Most Actives' ticker symbols from CNN Markets.
    Returns a deduplicated list of ticker strings.
    Falls back to a known high-volume list if the page cannot be scraped.
    """
    print(f"Launching headless Chrome -> {CNN_MARKETS_URL}")
    driver  = None
    tickers = []

    try:
        driver = build_chrome_driver()
        driver.get(CNN_MARKETS_URL)  # Navigate to CNN Markets

        candidate_selectors = [
            "[class*='ticker']",         
            "[class*='symbol']",         
            "[class*='market-mover']",   
            "[class*='mostActive']",    
            "[class*='most-active']",   
            "td.market-data__symbol",   
        ]

        wait = WebDriverWait(driver, SELENIUM_WAIT)

        for selector in candidate_selectors:
            try:
    
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
                elements = driver.find_elements(By.CSS_SELECTOR, selector)

                for el in elements:
                    text = el.text.strip()
                    # finds and matches ticker pattern
                    if re.match(r'^[A-Z]{1,5}$', text):
                        tickers.append(text)

                if tickers:
                    print(f"Strategy A succeeded with selector: '{selector}'")
                    break  # Stop trying selectors once we have valid tickers

            except TimeoutException:
                continue

        # After JS runs, driver.page_source includes dynamically-injected JSON.
        # Look for patterns like "symbol":"NVDA" or "ticker":"AAPL".
        if not tickers:
            print("Strategy A found nothing. Trying Strategy B (regex on rendered DOM)...")
            page_source = driver.page_source
            pattern     = r'"(?:symbol|ticker)"\s*:\s*"([A-Z]{1,5})"'
            found       = re.findall(pattern, page_source)

            seen = set()
            for t in found:
                if t not in seen:
                    seen.add(t)
                    tickers.append(t)

            if tickers:
                print(f"Strategy B found {len(tickers)} tickers via regex.")

    except WebDriverException as e:
        # Chrome failed to launch or crashed — fall through to the hardcoded fallback
        print(f"[ERROR] Selenium/Chrome error: {e}")

    finally:
        # Always quit the browser — releases the Chrome process and frees memory
        if driver:
            driver.quit()
            print("Browser closed.")

    # Hardcoded fallback
    # Used only if both Selenium strategies fail (e.g. Chrome not installed,
    # network blocked, or CNN significantly changed its page structure).
    if not tickers:
        print("WARNING: Live ticker scraping failed. Using fallback ticker list.")
        tickers = ["NVDA", "TSLA", "AAPL", "AMD", "AMZN",
                   "MSFT", "META", "GOOGL", "INTC", "BAC"]

    # Deduplicate while preserving the original order
    seen           = set()
    unique_tickers = []
    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    print(f"\nTickers retrieved ({len(unique_tickers)} total): {unique_tickers}")
    return unique_tickers


# Run the scraper and store the result
tickers = get_most_active_tickers()


def parse_numeric(value_str):
    """
    Convert a string like '4,521,300' or '27.45' to a Python float.
    Returns None if the value is missing, '--', or cannot be parsed.
    """
    if not value_str or value_str.strip() in ('--', 'N/A', ''):
        return None
    cleaned = value_str.replace(',', '').strip()  # Remove thousand-separator commas
    try:
        return float(cleaned)
    except ValueError:
        return None  # Non-numeric string — return None gracefully


def get_yahoo_data(ticker):
    """
    Fetch Open Price, Average Volume, and PE Ratio for a given ticker
    from Yahoo Finance using requests + BeautifulSoup.
    """
    url = f"{YAHOO_QUOTE_BASE}{ticker}"

    # Initialize all fields to None, safe defaults if parsing fails
    result = {
        "ticker":     ticker,
        "open_price": None,
        "avg_volume": None,
        "pe_ratio":   None,
    }

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()  # Raise exception for HTTP 4xx/5xx responses
    except requests.exceptions.RequestException as e:
        print(f"  [ERROR] Could not fetch {ticker}: {e}")
        return result  # Return None-filled dict on network error

    soup = BeautifulSoup(response.text, "html.parser")

    # Map exact label text from Yahoo Finance to our result dict keys
    label_map = {
        "Open":            "open_price",
        "Avg. Volume":     "avg_volume",
        "PE Ratio (TTM)": "pe_ratio",
    }

    # Yahoo Finance renders stats in a table where the label is in one <td>
    # and the value is in the immediately following sibling <td>.
    for td in soup.find_all("td"):
        label_text = td.get_text(strip=True)
        if label_text in label_map:
            value_td = td.find_next_sibling("td")  # Value is the next cell
            if value_td:
                key = label_map[label_text]
                result[key] = parse_numeric(value_td.get_text(strip=True))

    # Some page variants use <li> elements where the first <span> is the label
    # and the last <span> is the value.
    if all(result[k] is None for k in ["open_price", "avg_volume", "pe_ratio"]):
        for li in soup.find_all("li"):
            spans = li.find_all("span")
            if len(spans) >= 2:
                label_text = spans[0].get_text(strip=True)
                value_text = spans[-1].get_text(strip=True)
                if label_text in label_map:
                    key = label_map[label_text]
                    result[key] = parse_numeric(value_text)

    # Convert avg_volume to INTEGER as required by the project spec
    if result["avg_volume"] is not None:
        result["avg_volume"] = int(result["avg_volume"])

    return result


print("Yahoo Finance scraper defined. Ready to collect data.")

stock_data = []  # Accumulates one dict per ticker

print(f"Fetching Yahoo Finance data for {len(tickers)} tickers...\n")
print(f"{'Ticker':<8} {'Open Price':>12} {'Avg Volume':>16} {'PE Ratio':>10}")
print("-" * 52)

for ticker in tickers:
    data = get_yahoo_data(ticker)  # Handles its own exceptions internally
    stock_data.append(data)

    # Format for display, show 'N/A' for any missing values
    open_str = f"{data['open_price']:.2f}" if data['open_price'] is not None else "N/A"
    vol_str  = f"{data['avg_volume']:,}"   if data['avg_volume'] is not None else "N/A"
    pe_str   = f"{data['pe_ratio']:.2f}"  if data['pe_ratio']   is not None else "N/A"

    print(f"{ticker:<8} {open_str:>12} {vol_str:>16} {pe_str:>10}")

    # avoids triggering Yahoo Finance rate limiting
    time.sleep(CRAWL_DELAY)

print("-" * 52)
print(f"\nData collection complete. {len(stock_data)} records gathered.")


def write_text_file(data_list, filepath):
    """
    Write stock data to a comma-separated text file.

    Parameters
    ----------
    data_list : list of dicts
    filepath  : str — destination path
    """
    if not data_list:
        print("ERROR: No data available to write.")
        return

    try:
        with open(filepath, "w") as f:
            # Write the header row
            f.write("Ticker,OpenPrice,AvgVolume,PERatio\n")

            # Write one data row per ticker
            for row in data_list:
                # Convert None to empty string so the CSV stays clean
                ticker     = row["ticker"]
                open_price = row["open_price"] if row["open_price"] is not None else ""
                avg_volume = row["avg_volume"] if row["avg_volume"] is not None else ""
                pe_ratio   = row["pe_ratio"]   if row["pe_ratio"]   is not None else ""
                f.write(f"{ticker},{open_price},{avg_volume},{pe_ratio}\n")

        print(f"stocks.txt created: {os.path.abspath(filepath)}")
        print(f"Rows written (excluding header): {len(data_list)}")

    except IOError as e:
        print(f"ERROR: Could not write {filepath}: {e}")


# Write all stock data to stocks.txt
write_text_file(stock_data, TEXT_FILE)

# Print file contents for verification
print("\n--- stocks.txt contents ---")
with open(TEXT_FILE, "r") as f:
    print(f.read())
    
def create_and_populate_database(data_list, db_filepath, table_name):
    """
    Create StocksDatabase.db and insert all stock records into StocksTable.

    Parameters
    ----------
    data_list   : list of dicts
    db_filepath : str — path for the .db file
    table_name  : str — SQL table name
    """
    if not data_list:
        print("ERROR: No data to insert into the database.")
        return

    try:
        # Connect to (or create) the SQLite database file
        conn   = sqlite3.connect(db_filepath)
        cursor = conn.cursor()

        # Drop any leftover table then create a fresh one with the required schema
        cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
        cursor.execute(f"""
            CREATE TABLE {table_name} (
                Ticker    TEXT,
                OpenPrice REAL,
                AvgVolume INTEGER,
                PERatio   REAL
            )
        """)

        # Build list of tuples for bulk insert
        rows_to_insert = [
            (
                row["ticker"],
                row["open_price"],  # Python None automatically becomes SQL NULL
                row["avg_volume"],
                row["pe_ratio"],
            )
            for row in data_list
        ]

        # Parameterized INSERT — safe from SQL injection, handles None -> NULL
        cursor.executemany(
            f"INSERT INTO {table_name} (Ticker, OpenPrice, AvgVolume, PERatio) "
            f"VALUES (?, ?, ?, ?)",
            rows_to_insert
        )

        conn.commit()  # Persist all inserts to disk

        print(f"StocksDatabase.db created: {os.path.abspath(db_filepath)}")
        print(f"Table '{table_name}' created and populated with {len(rows_to_insert)} rows.")

        # ── Verification: read back all rows ──────────────────────────────────
        print(f"\n--- Verification: SELECT * FROM {table_name} ---")
        cursor.execute(f"SELECT * FROM {table_name}")
        rows      = cursor.fetchall()
        col_names = [desc[0] for desc in cursor.description]

        print(f"  {' | '.join(col_names)}")
        print("  " + "-" * 58)

        for r in rows:
            ticker     = r[0]
            open_price = f"{r[1]:.2f}" if r[1] is not None else "NULL"
            avg_volume = f"{r[2]:,}"   if r[2] is not None else "NULL"
            pe_ratio   = f"{r[3]:.2f}" if r[3] is not None else "NULL"
            print(f"  {ticker:<8} | {open_price:>10} | {avg_volume:>16} | {pe_ratio:>8}")

        conn.close()  # Release the database connection

    except sqlite3.Error as e:
        print(f"DATABASE ERROR: {e}")


# Create and populate the database with all collected stock data
create_and_populate_database(stock_data, DB_FILE, DB_TABLE)

print("=" * 58)
print("FINAL OUTPUT FILE VERIFICATION")
print("=" * 58)

for filepath in [TEXT_FILE, DB_FILE]:
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)  # Size in bytes
        print(f"  [OK]      {filepath:<25} {size} bytes")
        print(f"            {os.path.abspath(filepath)}")
    else:
        # Should never reach here if all prior cells ran without errors
        print(f"  [MISSING] {filepath} — file was not created!")
print("Project Execution Complete.")

All libraries imported successfully.
True
Text file will be saved as    : stocks.txt
Database file will be saved as: StocksDatabase.db
Launching headless Chrome -> https://www.cnn.com/markets
Strategy A found nothing. Trying Strategy B (regex on rendered DOM)...
Browser closed.

Tickers retrieved (10 total): ['NVDA', 'TSLA', 'AAPL', 'AMD', 'AMZN', 'MSFT', 'META', 'GOOGL', 'INTC', 'BAC']
Yahoo Finance scraper defined. Ready to collect data.
Fetching Yahoo Finance data for 10 tickers...

Ticker     Open Price       Avg Volume   PE Ratio
----------------------------------------------------
NVDA           221.59      151,325,148      34.24
TSLA           322.33       45,302,554     304.24
AAPL           311.32       56,809,803      35.97
AMD            497.94       31,047,395     123.31
AMZN           272.89       50,490,827      22.08
MSFT           499.21       41,354,858      27.85
META           585.64       18,166,272      22.32
GOOGL          356.77       33,177,020      17.76
INTC  